# Workshop 3 — ETL Streaming with Apache Kafka
## Step 1 & 2: Exploratory Data Analysis (EDA) + Data Cleaning & Harmonization

**Course:** ETL (G01) — Data Engineering and Artificial Intelligence  
**Dataset:** World Happiness Report 2015–2019

In [ ]:
import sys
!{sys.executable} -m pip install matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

---
## 1. Data Loading

In [ ]:
# Paths — adjust if your CSV files are in a different folder
DATA_PATH = '../data/raw/'

files = {
    2015: pd.read_csv(DATA_PATH + '2015.csv'),
    2016: pd.read_csv(DATA_PATH + '2016.csv'),
    2017: pd.read_csv(DATA_PATH + '2017.csv'),
    2018: pd.read_csv(DATA_PATH + '2018.csv'),
    2019: pd.read_csv(DATA_PATH + '2019.csv'),
}

for year, df in files.items():
    print(f'\n=== {year} ===')
    print(f'Shape: {df.shape}')
    print(f'Columns: {list(df.columns)}')

---
## 2. Schema Differences Between Years

The datasets **do not share the same schema**. Here we analyze which columns each year has.

In [ ]:
print('Columns per year:')
for year, df in files.items():
    print(f'\n{year}:')
    for col in df.columns:
        print(f'  - {col} ({df[col].dtype})')

---
## 3. Data Quality Analysis Per Year

### 3.1 Missing Values

In [ ]:
print('Missing values per column and year:')
for year, df in files.items():
    nulls = df.isnull().sum()
    if nulls.any():
        print(f'\n{year}:')
        print(nulls[nulls > 0])
    else:
        print(f'\n{year}: No missing values ✓')

### 3.2 Duplicated Records

In [ ]:
for year, df in files.items():
    dups = df.duplicated().sum()
    print(f'{year}: {dups} duplicates')

### 3.3 Descriptive Statistics

In [ ]:
for year, df in files.items():
    print(f'\n=== {year} ===')
    display(df.describe())

---
## 4. Schema Harmonization

**Design decision:** Each year uses different column names for the same concepts.
We create a unified mapping based on the semantic meaning of each column.

**Proposed unified schema:**

| Unified field | Description |
|---|---|
| `country` | Country |
| `year` | Report year |
| `happiness_score` | Happiness score (target) |
| `gdp` | GDP per capita |
| `family` | Social / family support |
| `health` | Healthy life expectancy |
| `freedom` | Freedom to make life choices |
| `generosity` | Generosity |
| `corruption` | Perception of corruption |

In [ ]:
# Mapping from original column names → unified schema per year
column_maps = {
    2015: {
        'Country': 'country',
        'Happiness Score': 'happiness_score',
        'Economy (GDP per Capita)': 'gdp',
        'Family': 'family',
        'Health (Life Expectancy)': 'health',
        'Freedom': 'freedom',
        'Trust (Government Corruption)': 'corruption',
        'Generosity': 'generosity',
    },
    2016: {
        'Country': 'country',
        'Happiness Score': 'happiness_score',
        'Economy (GDP per Capita)': 'gdp',
        'Family': 'family',
        'Health (Life Expectancy)': 'health',
        'Freedom': 'freedom',
        'Trust (Government Corruption)': 'corruption',
        'Generosity': 'generosity',
    },
    2017: {
        'Country': 'country',
        'Happiness.Score': 'happiness_score',
        'Economy..GDP.per.Capita.': 'gdp',
        'Family': 'family',
        'Health..Life.Expectancy.': 'health',
        'Freedom': 'freedom',
        'Trust..Government.Corruption.': 'corruption',
        'Generosity': 'generosity',
    },
    2018: {
        'Country or region': 'country',
        'Score': 'happiness_score',
        'GDP per capita': 'gdp',
        'Social support': 'family',
        'Healthy life expectancy': 'health',
        'Freedom to make life choices': 'freedom',
        'Perceptions of corruption': 'corruption',
        'Generosity': 'generosity',
    },
    2019: {
        'Country or region': 'country',
        'Score': 'happiness_score',
        'GDP per capita': 'gdp',
        'Social support': 'family',
        'Healthy life expectancy': 'health',
        'Freedom to make life choices': 'freedom',
        'Perceptions of corruption': 'corruption',
        'Generosity': 'generosity',
    },
}

UNIFIED_COLS = ['country', 'year', 'happiness_score', 'gdp', 'family', 'health', 'freedom', 'generosity', 'corruption']

harmonized_dfs = []

for year, df in files.items():
    mapping = column_maps[year]
    df_renamed = df.rename(columns=mapping)
    df_renamed['year'] = year
    # Select only columns from the unified schema
    available = [c for c in UNIFIED_COLS if c in df_renamed.columns]
    df_clean = df_renamed[available].copy()
    harmonized_dfs.append(df_clean)
    print(f'{year}: {df_clean.shape[0]} rows, available columns: {available}')

df_unified = pd.concat(harmonized_dfs, ignore_index=True)
print(f'\nUnified dataset: {df_unified.shape}')

---
## 5. Cleaning the Unified Dataset

In [ ]:
print('Missing values before cleaning:')
print(df_unified.isnull().sum())

# Strategy: impute numeric nulls with the median per year
numeric_cols = ['gdp', 'family', 'health', 'freedom', 'generosity', 'corruption']

for col in numeric_cols:
    if col in df_unified.columns:
        df_unified[col] = df_unified.groupby('year')[col].transform(
            lambda x: x.fillna(x.median())
        )

# Drop rows where the target or country identifier is missing
df_unified.dropna(subset=['happiness_score', 'country'], inplace=True)

# Normalize country name
df_unified['country'] = df_unified['country'].str.strip()

print('\nMissing values after cleaning:')
print(df_unified.isnull().sum())
print(f'\nFinal shape: {df_unified.shape}')

---
## 6. EDA Visualizations

In [ ]:
# Happiness score distribution
plt.figure(figsize=(10, 4))
sns.histplot(df_unified['happiness_score'], bins=30, kde=True, color='steelblue')
plt.title('Happiness Score Distribution (2015–2019)')
plt.xlabel('Happiness Score')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
corr_cols = ['happiness_score', 'gdp', 'family', 'health', 'freedom', 'generosity', 'corruption']
corr_df = df_unified[corr_cols].dropna()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: features vs happiness score
features = ['gdp', 'family', 'health', 'freedom']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, feat in zip(axes.flatten(), features):
    ax.scatter(df_unified[feat], df_unified['happiness_score'], alpha=0.4, color='steelblue')
    ax.set_xlabel(feat)
    ax.set_ylabel('happiness_score')
    ax.set_title(f'{feat} vs happiness_score')

plt.suptitle('Feature Relationships vs Happiness Score', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot: happiness score per year
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_unified, x='year', y='happiness_score', palette='Set2')
plt.title('Happiness Score per Year')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 happiest countries (average 2015–2019)
top10 = df_unified.groupby('country')['happiness_score'].mean().nlargest(10).reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=top10, x='happiness_score', y='country', palette='Blues_r')
plt.title('Top 10 Happiest Countries (Average 2015–2019)')
plt.xlabel('Average Happiness Score')
plt.tight_layout()
plt.show()

---
## 7. Outlier Detection

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
cols_to_check = ['happiness_score', 'gdp', 'family', 'health', 'freedom', 'generosity', 'corruption']

for ax, col in zip(axes.flatten(), cols_to_check):
    if col in df_unified.columns:
        sns.boxplot(y=df_unified[col], ax=ax, color='lightcoral')
        ax.set_title(col)

axes.flatten()[-1].set_visible(False)
plt.suptitle('Outliers per Variable', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Outlier report using IQR method
print('\nOutlier count per variable (IQR method):')
for col in cols_to_check:
    if col in df_unified.columns:
        Q1 = df_unified[col].quantile(0.25)
        Q3 = df_unified[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers = ((df_unified[col] < Q1 - 1.5*IQR) | (df_unified[col] > Q3 + 1.5*IQR)).sum()
        print(f'  {col}: {outliers} outliers')

---
## 8. Data Quality Observations

| Problem | Affected Year(s) | Decision |
|---|---|---|
| Different column names for the same concept | All years | Unified mapping per year |
| Extra columns (rank, region, etc.) | 2015, 2016, 2017 | Dropped — not relevant for modeling |
| Null values in `corruption` | 2018 | Imputed with median per year |
| Inconsistent data types | Some years | Implicitly cast to float64 by pandas |
| Mild outliers | Generosity, Corruption | Kept — real values from the official report |

---
## 9. Export Unified Dataset

In [ ]:
output_path = '../data/processed/happiness_unified.csv'
df_unified.to_csv(output_path, index=False)
print(f'Unified dataset saved to: {output_path}')
print(f'Shape: {df_unified.shape}')
display(df_unified.head(10))